# Hypercube family — full exploration, n = 1, ..., 6

For each n (the hypercube's dimension, so it has $2^n$ vertices — matching the convention in `hypercube.sage`), this notebook:

1. lists all vertices,
2. computes the canonical form via the general nbc method (Brown–Dupont Prop. 6.7, `general_canonical_forms.sage`), checking the defining pole-structure property,
3. computes the projective (polar) dual,
4. checks the **volume conjecture**: the canonical form, evaluated at the centroid (in a chart re-centered there), equals ±n! times the volume of the projective dual taken at that same centroid — with a short demonstration (n=1 section) of why it has to be the centroid and not an arbitrary interior point,
5. enumerates all triangulations and identifies which are regular (n = 1, 2, 3 only — see the n=4-6 section for why),
6. computes the secondary polytope and its vertex embedding (n = 1, 2, 3 only).

Same structure as `simplex_explorer.ipynb` — see that notebook for more discussion of each step. The hypercube is a genuinely different test of the same methods: unlike the simplex, it's *not* its own only triangulation (the cube has 74 of them, and the count explodes further from there), so this is the first notebook in this series where the triangulation/secondary-polytope section has non-trivial content — and also where it hits a hard computational ceiling well before n=6.

**Requires the `sagemath` Jupyter kernel** and must be opened from the same synced folder as the `.sage` files — see `README.md`.

In [ ]:
load("general_canonical_forms.sage")

That `load` pulls in `common.sage` (vertex generators, `polar_dual`, `secondary_polytope_data`) and `vertex_sum_canonical_forms.sage` too, and runs `general_canonical_forms.sage`'s own test suite as a side effect (scroll up for that PASS/FAIL output). Everything below is fresh, per-instance exploration of the hypercube family specifically.

## n = 1 — the segment

In [ ]:
n = 1
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = hypercube_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Hypercube n={n}", phi, P, y)
phi

### Canonical form, broken down by vertex

The hypercube is simple, so every vertex has valency exactly n and contributes exactly one nbc term — the whole-polytope expression above already *is* essentially this sum. `canonical_form_by_vertex` (`common.sage`) is still worth running: it's the same code path used on the non-simple families elsewhere in this series (`cross_polytope_explorer.ipynb`, `hypersimplex_explorer.ipynb`, `cyclic_polytope_explorer.ipynb`), so seeing it collapse to "1 term per vertex" here is a direct, visible confirmation that the general method reduces to Proposition 6.10's one-term-per-vertex formula on a simple polytope.

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual

The volume conjecture (Arkani-Hamed–Bai–Lam) relates the canonical form not to the polytope's own volume, but to the volume of its **projective dual**. This needs care about *where* the dual is taken from: the ordinary (metric) polar dual $P^\circ = \{y : x\cdot y \le 1\ \forall x \in P\}$ depends on an arbitrary choice of origin, and its volume is wildly sensitive to that choice (demonstrated two cells down, for an origin with no particular geometric meaning). The *projective* dual fixes this by using the one reference point every polytope determines from its own vertices with no external choice involved — the **centroid**. Once the canonical form is re-expressed in a chart centered there, and the dual is taken there too, the identity is exact:

$$\phi_{\text{centroid}}(0) = \pm\, n!\cdot\mathrm{Vol}\!\left(P^\circ_{\text{centroid}}\right)$$

with the same overall orientation sign built into the canonical form's own definition. `common.sage`'s `polar_dual` already centers at the centroid internally, so `Dual = polar_dual(P)` from the previous cell already *is* $P^\circ_{\text{centroid}}$ — for the hypercube this dual is (combinatorially) the cross-polytope, matching the duality already stated on the site.

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

# Re-express the canonical form in the centroid-centered chart -- the
# original chart's y=0 is one of the hypercube's own vertices (a pole),
# not an interior point, so phi can't be evaluated at 0 there.
phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

#### Why it has to be the centroid, not just any interior point

Picking a different interior reference point breaks the identity — confirming this isn't a coincidence of using *some* origin near the polytope, but specifically requires the centroid (the one projectively canonical choice, with no external rescaling ambiguity). Below, an arbitrary nearby interior point is used instead — deliberately calling `.polar()` directly (bypassing `polar_dual`, which always re-centers at *its own* centroid) so the dual really is taken with respect to this off-centroid point.

In [ ]:
off_center = [c + QQ(1) / 5 for c in centroid]  # nudged off the centroid, still interior
pts_off = [tuple(QQ(v[i]) - off_center[i] for i in range(n)) for v in pts]
P_off = Polyhedron(vertices=pts_off)
assert P_off.interior_contains(vector([0] * n)), "sanity check: still an interior point"

phi_off = general_canonical_form_density(P_off, y)
val_off = phi_off.subs({yi: 0 for yi in y})
vol_dual_off = P_off.polar().volume()  # dual w.r.t. THIS (off-centroid) origin, not re-centered

print("phi at an off-centroid interior point =", val_off)
print("n! * Vol(dual there) =", factorial(n) * vol_dual_off)
print("these do not match -- the identity is specific to the centroid, not just any origin")

### All triangulations, and which are regular

The segment has exactly one triangulation (itself). See `secondary_polytope_data`'s docstring in `common.sage` for the method: matching each triangulation's GKZ vector against the secondary polytope's vertex list, entirely avoiding a TOPCOM subprocess call that was found to hang and crash WSL on this machine.

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

`pc.secondary_polytope()` naturally lives in R^(number of points) — one coordinate per point of the configuration — even though its actual dimension is usually much smaller; `reduce_secondary_polytope` (`common.sage`) projects onto its own affine hull via `Polyhedron.affine_hull_projection()`, giving vertex coordinates in exactly `sp.dimension()`-many numbers instead of a padded, redundant embedding.

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 2 — the square

In [ ]:
n = 2
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = hypercube_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Hypercube n={n}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

The square has exactly 2 triangulations (split along either diagonal) — both expected to be regular (a standard fact for convex quadrilaterals), checked here rather than assumed.

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## n = 3 — the cube

In [ ]:
n = 3
y = [var(f"y{i}") for i in range(1, n + 1)]
pts = hypercube_vertices(n)
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Hypercube n={n}", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
print("f-vector:", Dual.f_vector())
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid — see the n=1 section for why)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()  # Dual = polar_dual(P), already centroid-centered (previous cell)
target = factorial(n) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("n! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches n! * Vol(dual):", match_plus, " matches -n! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

The cube is the first genuinely non-trivial case in this notebook series: it has many triangulations, not just one or two. `secondary_polytope_data` enumerates them all via Sage's internal engine (fast even here — well under a second) and checks regularity by GKZ-vector matching against the secondary polytope, rather than via TOPCOM's own (WSL-crashing, on this machine) regularity test — see `common.sage` and `README.md`.

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")
rows[:5]

### Secondary polytope: vertex embedding

One vertex per *regular* triangulation. The raw `pc.secondary_polytope()` lives in $\mathbb{R}^8$ (one coordinate per vertex of the cube) even though it's only 4-dimensional; `reduce_secondary_polytope` projects it onto its own affine hull first, so the vertices printed below are genuine 4-tuples, not padded 8-tuples.

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()[:5]

## n = 4, 5, 6 — canonical form, dual, and the volume conjecture only

Unlike the simplex, the hypercube's triangulation count explodes fast: the 3-cube already has 74 triangulations, and the 4-cube is well documented in the literature to have tens of millions — completely infeasible to enumerate with `secondary_polytope_data` (or any exhaustive method). Canonical form computation itself has no such problem here (the hypercube is simple, so there's exactly one nbc term per vertex, and $2^n$ grows far slower than the triangulation count) — n=6 takes under a second — so this section covers n=4, 5, 6 for steps 1–4 only, looped rather than repeated cell-by-cell since there's nothing new to say about each beyond the numbers.

In [ ]:
for n in [4, 5, 6]:
    y = [var(f"y{i}") for i in range(1, n + 1)]
    pts = hypercube_vertices(n)
    P = Polyhedron(vertices=pts)

    phi = general_canonical_form_density(P, y)
    pole_ok = verify_pole_structure(f"Hypercube n={n}", phi, P, y)

    Dual = polar_dual(P)

    centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(n)]
    pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(n)) for v in pts]
    P_centered = Polyhedron(vertices=pts_centered)
    phi_centroid = general_canonical_form_density(P_centered, y)
    val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})
    target = factorial(n) * Dual.volume()
    vol_ok = bool((val_at_centroid - target) == 0) or bool((val_at_centroid + target) == 0)

    print(f"n={n}: {len(pts)} vertices, dual has {Dual.n_vertices()} vertices, "
          f"pole structure {'PASS' if pole_ok else 'FAIL'}, volume conjecture {'PASS' if vol_ok else 'FAIL'}")
    assert pole_ok and vol_ok, f"n={n}: a real check failed"